# Fields to update
1. MB_ID (Done)
2. Country
3. Sub_genre
4. Homepage
5. Bandcamp page

In [1]:
import sys
import os

sys.path.append(os.path.abspath('..'))

In [2]:
%pip install -r requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: c:\Users\dsvel\.pyenv\pyenv-win\versions\3.12.9\python.exe -m pip install --upgrade pip


In [3]:
import sys
print(sys.version)
print(sys.executable)

3.12.9 (tags/v3.12.9:fdb8142, Feb  4 2025, 15:27:58) [MSC v.1942 64 bit (AMD64)]
c:\Users\dsvel\.pyenv\pyenv-win\versions\3.12.9\python.exe


In [4]:
# Notion imports
from Notion.reader import get_artists_db
from Notion.writer import build_property, update_artist
from Notion.database import myNotionDatabases
from Data.artists import Artist
from Data.tags import clean_tags
from Datasources.Last_Fm import get_artist_tags
# MusicBrainz imports
from Datasources.MusicBrainz import search_artist,get_artist_info, load_cache, save_cache
from tqdm import tqdm
import random
import copy

In [5]:
notion_response = get_artists_db()
#artists_db = []
db = myNotionDatabases()
Updated_notion_artist_database = {}

for artist in notion_response:
    extracted_artist = db.extract_artist(artist)
    #artists_db.append(extracted_artist)
    key = extracted_artist['artist_name'].lower().strip()
    Updated_notion_artist_database[key] = Artist(
        name=extracted_artist['artist_name'],
        mbid=extracted_artist['mb_id'],
        country=extracted_artist['country'],
        tags=extracted_artist.get('subgenre', []),
        website_url=extracted_artist.get('website'),
        bandcamp_url=extracted_artist.get('bandcamp_url'),
        page_id=extracted_artist.get('id')
    )


Original_notion_artist_database = copy.deepcopy(Updated_notion_artist_database)

In [ ]:
#for artist in random.sample(sorted(notion_artist_database.values(), key=lambda a: a.name), min(10, len(notion_artist_database))): #Selecting X random artists for testing purposes
no_matches = []
for artist in tqdm(Updated_notion_artist_database.values()):
    if artist.mbid is None:
        try:
            mb_id, match_result = search_artist(artist_name=artist.name.strip(), confidence_threshold=85)
            if mb_id is None:
                #this will force an update and send the result to the notion database. 
                # I can then make a view which filters for them.
                artist.mbid = f"No match found for |{artist.name}| with the reason: {match_result}"
            else:
                mb_artist_info = get_artist_info(mb_id)
                artist.mbid = mb_id
                artist.country = mb_artist_info.get("country")
                artist.website_url = mb_artist_info.get("homepage")
                artist.bandcamp_url = mb_artist_info.get("bandcamp")
                artist.tags = mb_artist_info.get("tags", [])
        except Exception as e:
            print(f"Error processing {artist.name}: {e}")



100%|██████████| 116/116 [00:01<00:00, 79.24it/s]


In [7]:
for error in no_matches:
    print(error)
print(f"Number of missing matches = {len(no_matches)}")
no_exact_match = len([x for x in no_matches if "No Exact Match" in x])
print(f"No Exact Match Issues = {no_exact_match}")
multi_exact_match = len([x for x in no_matches if "Multiple Exact Matches" in x])
print(f"Multiple Exact Match Issues = {multi_exact_match}")

No match found for |Hammer King| with the reason: No Match — best fuzzy was 'carole king' (64%)
No match found for |65 Days of Static| with the reason: No Match — best fuzzy was 'eat static' (59%)
No match found for |The Browning| with the reason: No Match — best fuzzy was 'the rolling stones' (60%)
No match found for |The Hu| with the reason: No Match — best fuzzy was 'the who' (77%)
No match found for |Baby Metal| with the reason: No Match — best fuzzy was 'baby bash' (63%)
No match found for |Marc Hudson| with the reason: No Match — best fuzzy was 'marc almond' (64%)
No match found for |Hanabie| with the reason: No results above confidence threshold
No match found for |Spinal Tap| with the reason: No Match — best fuzzy was 'spinal' (75%)
No match found for |King Kraken| with the reason: No Match — best fuzzy was 'king crimson' (61%)
No match found for |The Amazons| with the reason: No Match — best fuzzy was 'the kinks' (60%)
Number of missing matches = 10
No Exact Match Issues = 0
M

In [8]:
#Get the Last.fm info for the artist.
last_fm_artists = []
for artist in Updated_notion_artist_database.values():
    if artist.mbid is not None:
        last_fm_artists.append(artist)

In [9]:
for artist in tqdm(last_fm_artists):
    #print(artist.name)
    last_fm_response = get_artist_tags(mb_id=artist.mbid)
    artist.add_tags(last_fm_response)

100%|██████████| 106/106 [00:14<00:00,  7.12it/s]


In [10]:
for artist in tqdm(last_fm_artists):
    cleaned_tags = clean_tags(artist.tags)
    sorted_tags = sorted(cleaned_tags.items(), key=lambda x: x[1], reverse=True)
    Top_x = sorted_tags[:3]
    declared_subgenres = []

    for tag in Top_x:
        declared_subgenres.append(tag)
    artist.tags = Top_x

100%|██████████| 106/106 [00:00<00:00, 105982.41it/s]


In [ ]:
for key, original in Original_notion_artist_database.items():
    updated_data = Updated_notion_artist_database[key]
    updates = {}
    page_id = original.page_id
    if updated_data.mbid is None:
        updates["MB_ID"] = ("rich_text", updated_data.mbid)
    else:
    if original.mbid is None and updated_data.mbid != None:
        updates["MB_ID"] = ("rich_text", updated_data.mbid)
    if original.country is None and updated_data.country != None:
        updates["Country"] = ("rich_text", updated_data.country)
    if original.bandcamp_url is None and updated_data.bandcamp_url != None:
        updates["Bandcamp url"] = ("url", updated_data.bandcamp_url)
    if original.website_url is None and updated_data.website_url != None:
        updates["Website"] = ("url", updated_data.website_url)
    
    if len(updates) != 0:
        update_artist(page_id=page_id, updates=updates, dry_run=True)#This line acts as a printout
        update_artist(page_id=page_id, updates=updates, dry_run=False)

DRY RUN — 3396e950-cac6-8003-8846-db9b58c926f5:
  MB_ID: {'rich_text': [{'text': {'content': '89e39f67-65cc-4f90-b145-b1b56c209f8a'}}]}
  Website: {'url': 'http://www.craigdavid.com/'}
DRY RUN — 3396e950-cac6-800a-9611-e094e9be0f49:
  MB_ID: {'rich_text': [{'text': {'content': '319b1175-ced9-438f-986b-9239c3edd92d'}}]}
  Website: {'url': 'https://www.sonataarctica.info/'}
DRY RUN — 3396e950-cac6-800a-9824-ccb61d1293c4:
  MB_ID: {'rich_text': [{'text': {'content': '4a00ec9d-c635-463a-8cd4-eb61725f0c60'}}]}
  Bandcamp url: {'url': 'https://deadmau5.bandcamp.com/'}
  Website: {'url': 'http://www.deadmau5.com/'}
DRY RUN — 3396e950-cac6-800e-8519-c2858a8d4091:
  MB_ID: {'rich_text': [{'text': {'content': '3b0e8f01-3fd9-4104-9532-1e4b526ce562'}}]}
  Bandcamp url: {'url': 'https://delain.bandcamp.com/'}
  Website: {'url': 'http://www.delain.nl/'}
DRY RUN — 3396e950-cac6-8011-bc2f-f37d0205c912:
  MB_ID: {'rich_text': [{'text': {'content': 'e4549ab4-48e1-4e0d-8c36-cb077f680b67'}}]}
  Website: {